# Emotion-Aware Multi-Artist Art Generation

**Pipeline:**
1. User writes how they feel (Arabic or English)
2. Emotion Detection Model maps text → Joy / Calm / Sadness
3. Stable Diffusion generates an artwork in the selected artist's style
4. Claude AI describes the generated image in detail

> Make sure you are on a **GPU runtime** in Colab: Runtime → Change runtime type → T4 GPU

## Step 1 — Install Dependencies

In [ ]:
!pip install diffusers transformers accelerate sentencepiece \
             anthropic torch torchvision xformers -q

import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch to GPU runtime")

## Step 2 — Emotion Detection

We use a pre-trained multilingual emotion classifier from HuggingFace.
It accepts Arabic and English text and maps it to one of our three emotions: **Joy, Calm, Sadness**.

The model used is `j-hartmann/emotion-english-distilroberta-base` for English
and we translate Arabic input first using `Helsinki-NLP/opus-mt-ar-en`.

In [ ]:
from transformers import pipeline

# Arabic → English translator
print("Loading translation model...")
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-ar-en",
    device=0 if torch.cuda.is_available() else -1
)

# Emotion classifier (7 classes)
print("Loading emotion classifier...")
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

print("Models loaded.")

In [ ]:
# Mapping from the model's 7 emotion labels to our 3 project labels
EMOTION_MAP = {
    "joy"      : "Joy",
    "surprise" : "Joy",
    "neutral"  : "Calm",
    "disgust"  : "Calm",
    "fear"     : "Sadness",
    "sadness"  : "Sadness",
    "anger"    : "Sadness",
}

def detect_emotion(user_text: str) -> dict:
    """
    Takes user text in Arabic or English.
    Returns detected emotion label and confidence score.
    """
    # Detect if Arabic (contains Arabic Unicode range)
    is_arabic = any("\u0600" <= c <= "\u06FF" for c in user_text)

    if is_arabic:
        translated = translator(user_text, max_length=128)[0]["translation_text"]
        print(f"Translated: '{user_text}' → '{translated}'")
        text_for_model = translated
    else:
        text_for_model = user_text

    # Run emotion classification
    results   = emotion_classifier(text_for_model)[0]
    top       = max(results, key=lambda x: x["score"])
    raw_label = top["label"].lower()
    mapped    = EMOTION_MAP.get(raw_label, "Calm")

    return {
        "input_text"    : user_text,
        "raw_emotion"   : raw_label,
        "emotion_label" : mapped,
        "confidence"    : round(top["score"], 4),
        "all_scores"    : {r["label"]: round(r["score"], 4) for r in results},
    }


# ── Quick test ──
test_cases = [
    "أنا حاسس بحزن شديد",
    "I feel amazing and full of energy",
    "كل حاجة تمام ومش حاسس بأي حاجة",
]

for t in test_cases:
    result = detect_emotion(t)
    print(f"Input: {result['input_text']}")
    print(f"  Emotion: {result['emotion_label']}  (raw: {result['raw_emotion']}, confidence: {result['confidence']})")
    print()

## Step 3 — Load Stable Diffusion Model

We use `stabilityai/stable-diffusion-2-1` via the HuggingFace `diffusers` library.

No training required — we control the artistic style and emotion through **prompt engineering**.
Each combination of artist + emotion gets a carefully crafted prompt that guides the generation.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

MODEL_ID = "stabilityai/stable-diffusion-2-1"

print("Loading Stable Diffusion model (this takes ~2 minutes on first run)...")

sd_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)
sd_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sd_pipe.scheduler.config)
sd_pipe = sd_pipe.to("cuda")

# Memory optimization for Colab T4
sd_pipe.enable_attention_slicing()

print("Stable Diffusion loaded and ready.")

## Step 4 — Prompt Engineering

The prompt is the bridge between the user's emotion and the generated image.
We craft a specific prompt for each artist × emotion combination.

Each prompt contains three layers:
- **Artist style descriptor** — unique visual characteristics of that artist
- **Emotion descriptor** — colors, mood, and composition that reflect the emotion
- **Quality tags** — guide the model toward high-quality output

In [ ]:
# Artist style descriptors — based on their actual visual characteristics
ARTIST_STYLES = {
    "Van Gogh": (
        "in the style of Vincent van Gogh, post-impressionist oil painting, "
        "thick impasto brushstrokes, swirling dynamic textures, "
        "vivid expressive colors, bold outlines"
    ),
    "Monet": (
        "in the style of Claude Monet, impressionist oil painting, "
        "soft loose brushwork, light reflections on water, "
        "atmospheric perspective, dappled light and color"
    ),
    "Leonardo da Vinci": (
        "in the style of Leonardo da Vinci, Renaissance oil painting, "
        "sfumato technique, subtle chiaroscuro, "
        "masterful composition, earthy muted palette"
    ),
}

# Emotion descriptors — colors, mood, and visual metaphors
EMOTION_PROMPTS = {
    "Joy": (
        "joyful and radiant, warm golden sunlight, vibrant yellows and oranges, "
        "blooming flowers, open sky, uplifting and celebratory mood, "
        "bright and luminous, full of life and energy"
    ),
    "Calm": (
        "calm and serene, peaceful landscape, soft muted tones, "
        "still water reflection, gentle morning light, "
        "tranquil and meditative atmosphere, balanced composition"
    ),
    "Sadness": (
        "melancholic and sorrowful, dark muted blues and grays, "
        "overcast sky, bare trees, solitary figure, "
        "heavy emotional weight, somber and introspective mood, "
        "deep shadows"
    ),
}

# Negative prompt — things we always want to avoid
NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, ugly, "
    "watermark, text, signature, modern photography, "
    "cartoon, anime, 3d render"
)

def build_prompt(artist: str, emotion: str) -> str:
    style   = ARTIST_STYLES[artist]
    emo     = EMOTION_PROMPTS[emotion]
    quality = "masterpiece, highly detailed, museum quality artwork, 8k"
    return f"{emo}, {style}, {quality}"


# Preview all 9 prompts (3 artists x 3 emotions)
for artist in ARTIST_STYLES:
    for emotion in EMOTION_PROMPTS:
        print(f"[{artist}] [{emotion}]")
        print(f"  {build_prompt(artist, emotion)[:120]}...")
        print()

## Step 5 — Image Generation Function

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def generate_artwork(
    artist: str,
    emotion: str,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    seed: int = 42,
) -> Image.Image:
    """
    Generate an artwork image given an artist name and emotion label.

    Parameters
    ----------
    artist              : One of 'Van Gogh', 'Monet', 'Leonardo da Vinci'
    emotion             : One of 'Joy', 'Calm', 'Sadness'
    num_inference_steps : More steps = higher quality but slower (30 is a good balance)
    guidance_scale      : How strictly to follow the prompt (7-9 is standard)
    seed                : Fixed seed for reproducibility

    Returns
    -------
    PIL Image of the generated artwork
    """
    prompt    = build_prompt(artist, emotion)
    generator = torch.Generator(device="cuda").manual_seed(seed)

    print(f"Generating: [{artist}] [{emotion}]")
    print(f"Prompt: {prompt[:100]}...")

    result = sd_pipe(
        prompt              = prompt,
        negative_prompt     = NEGATIVE_PROMPT,
        num_inference_steps = num_inference_steps,
        guidance_scale      = guidance_scale,
        generator           = generator,
        height              = 512,
        width               = 512,
    )

    image = result.images[0]
    print("Generation complete.")
    return image


def display_artwork(image: Image.Image, artist: str, emotion: str):
    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.title(f"{artist}  |  {emotion}", fontsize=14, fontweight="bold", pad=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## Step 6 — AI Image Description via Claude

After generating the image, we send it to the Claude API.
Claude analyzes the visual elements and writes a detailed interpretation
connecting the artistic choices back to the user's emotion.

**To get your API key:** https://console.anthropic.com

In [ ]:
import anthropic, base64, io

# Paste your Anthropic API key here
ANTHROPIC_API_KEY = "your-api-key-here"

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def image_to_base64(image: Image.Image) -> str:
    """Convert PIL Image to base64 string for the API."""
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


def describe_artwork(
    image: Image.Image,
    artist: str,
    emotion: str,
    user_text: str,
) -> str:
    """
    Send the generated image to Claude and get a detailed emotional interpretation.

    Parameters
    ----------
    image      : The generated PIL Image
    artist     : Artist name used for generation
    emotion    : Detected emotion label
    user_text  : Original text the user typed

    Returns
    -------
    String containing Claude's interpretation
    """
    img_b64 = image_to_base64(image)

    system_prompt = (
        "You are an expert art critic and emotion analyst. "
        "You analyze AI-generated artworks and explain how their visual elements "
        "reflect specific human emotions. "
        "Your descriptions are vivid, specific, and emotionally resonant. "
        "Always connect visual details (colors, brushstrokes, composition, light) "
        "directly to the emotional experience."
    )

    user_prompt = (
        f"The user expressed: '{user_text}'\n"
        f"Detected emotion: {emotion}\n"
        f"Generated in the style of: {artist}\n\n"
        f"Please analyze this generated artwork in detail. Explain:\n"
        f"1. The dominant colors and what emotional role they play\n"
        f"2. The brushwork or texture and how it reflects {emotion}\n"
        f"3. The overall composition and mood\n"
        f"4. How this image connects to what the user expressed: '{user_text}'\n\n"
        f"Write 3-4 paragraphs. Be specific about what you see in the image."
    )

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=system_prompt,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type"       : "base64",
                        "media_type" : "image/png",
                        "data"       : img_b64,
                    },
                },
                {"type": "text", "text": user_prompt},
            ],
        }],
    )

    return response.content[0].text

## Step 7 — Full Pipeline: Text → Emotion → Art → Description

This single function runs the entire pipeline end to end.

In [ ]:
def run_pipeline(
    user_text : str,
    artist    : str = None,
    seed      : int = 42,
) -> dict:
    """
    Full pipeline from user text to generated artwork and description.

    Parameters
    ----------
    user_text : What the user typed (Arabic or English)
    artist    : Artist name, or None to auto-select based on emotion
    seed      : Random seed for reproducibility

    Returns
    -------
    dict with keys: emotion_result, artist, image, description
    """

    print("=" * 60)
    print("STEP 1 — Detecting emotion...")
    print("=" * 60)

    emotion_result = detect_emotion(user_text)
    emotion_label  = emotion_result["emotion_label"]

    print(f"Detected emotion : {emotion_label}")
    print(f"Confidence       : {emotion_result['confidence']}")

    # Auto-select artist based on emotion if not specified
    # Van Gogh suits strong emotions, Monet suits calm, Leonardo suits sadness
    AUTO_ARTIST_MAP = {
        "Joy"    : "Van Gogh",
        "Calm"   : "Monet",
        "Sadness": "Leonardo da Vinci",
    }

    if artist is None:
        artist = AUTO_ARTIST_MAP[emotion_label]
        print(f"Auto-selected artist: {artist}")
    else:
        print(f"Artist (user selected): {artist}")

    print()
    print("=" * 60)
    print("STEP 2 — Generating artwork...")
    print("=" * 60)

    image = generate_artwork(artist, emotion_label, seed=seed)
    display_artwork(image, artist, emotion_label)

    print()
    print("=" * 60)
    print("STEP 3 — Generating AI description...")
    print("=" * 60)

    description = describe_artwork(image, artist, emotion_label, user_text)

    print()
    print("AI INTERPRETATION")
    print("-" * 60)
    print(description)
    print("-" * 60)

    return {
        "emotion_result" : emotion_result,
        "artist"         : artist,
        "image"          : image,
        "description"    : description,
    }

## Step 8 — Run the Pipeline

In [ ]:
# ── Example 1: Arabic input, auto-select artist ──
result = run_pipeline(
    user_text = "أنا حاسس بحزن شديد وكل حاجة حواليا تقيلة",
    artist    = None,    # auto-select based on emotion
    seed      = 42,
)

In [ ]:
# ── Example 2: English input, user picks artist ──
result2 = run_pipeline(
    user_text = "I feel peaceful and at ease with everything around me",
    artist    = "Monet",
    seed      = 7,
)

In [ ]:
# ── Example 3: Joy with Van Gogh ──
result3 = run_pipeline(
    user_text = "أنا سعيد جداً النهارده، كل حاجة تمام",
    artist    = "Van Gogh",
    seed      = 123,
)

## Step 9 — Generate All 9 Combinations (3 Artists x 3 Emotions)

This generates a visual grid showing all artist-emotion combinations.
Useful for the project presentation and LinkedIn showcase.

In [ ]:
import os
os.makedirs("/content/generated_artworks", exist_ok=True)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle("Emotion-Aware Multi-Artist Art Generation\n3 Artists x 3 Emotions",
             fontsize=16, fontweight="bold", y=1.01)

artists  = ["Van Gogh", "Monet", "Leonardo da Vinci"]
emotions = ["Joy", "Calm", "Sadness"]

EMOTION_COLORS = {"Joy": "#f5c518", "Calm": "#4a90d9", "Sadness": "#7b4f9e"}

for row_idx, emotion in enumerate(emotions):
    for col_idx, artist in enumerate(artists):

        img = generate_artwork(artist, emotion, seed=row_idx*10 + col_idx)

        # Save individual image
        fname = f"/content/generated_artworks/{artist.replace(' ', '_')}_{emotion}.png"
        img.save(fname)

        # Add to grid
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(
            f"{artist}\n{emotion}",
            fontsize=10, fontweight="bold",
            color=EMOTION_COLORS[emotion]
        )
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
grid_path = "/content/generated_artworks/all_combinations_grid.png"
plt.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print("Grid saved to:", grid_path)

## Step 10 — Interactive Demo Widget

A simple interactive widget for the demo / presentation.
Run this cell to get a live input box.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# UI components
text_input = widgets.Textarea(
    placeholder="Type how you feel in Arabic or English...",
    layout=widgets.Layout(width="500px", height="80px")
)

artist_dropdown = widgets.Dropdown(
    options=["Auto (AI decides)", "Van Gogh", "Monet", "Leonardo da Vinci"],
    description="Artist:",
    style={"description_width": "initial"},
)

generate_btn = widgets.Button(
    description="Generate Artwork",
    button_style="primary",
    layout=widgets.Layout(width="200px", height="40px")
)

output_area = widgets.Output()

def on_generate(btn):
    with output_area:
        clear_output(wait=True)
        user_text = text_input.value.strip()
        if not user_text:
            print("Please type something first.")
            return
        selected_artist = None if artist_dropdown.value == "Auto (AI decides)" \
                          else artist_dropdown.value
        run_pipeline(user_text=user_text, artist=selected_artist)

generate_btn.on_click(on_generate)

display(
    widgets.VBox([
        widgets.HTML("<h3>Emotion-Aware Art Generator</h3>"),
        widgets.HTML("<b>How are you feeling?</b>"),
        text_input,
        artist_dropdown,
        generate_btn,
        output_area,
    ])
)